In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold = spark.table("launch_reliability.gold.launch_features")

## overview of data

In [0]:
display(gold.agg(
    F.count("*").alias("launches"),
    F.sum(F.col("failed").cast("int")).alias("failures"),
    F.round(F.avg(F.col("failed").cast("int")) * 100, 1).alias("failure_pct"),
    F.sum(F.col("failed_strict").cast("int")).alias("failures_strict"),
    F.min("launch_year").alias("first_year"),
    F.max("launch_year").alias("last_year"),
    F.countDistinct("lv_type").alias("rocket_types"),
    F.countDistinct("lv_family").alias("families"),
    F.countDistinct("agency").alias("providers"),
))

## RQ1: How does failure risk drop as a vehicle accumulates flights?

In [0]:
display(gold
    .withColumn("flight_bucket",
        F.when(F.col("vehicle_flight_num") == 1, "1")
         .when(F.col("vehicle_flight_num") <= 3, "2-3")
         .when(F.col("vehicle_flight_num") <= 10, "4-10")
         .when(F.col("vehicle_flight_num") <= 30, "11-30")
         .otherwise("31+"))
    .groupBy("flight_bucket")
    .agg(F.count("*").alias("launches"),
         F.round(F.avg(F.col("failed").cast("int")), 3).alias("failure_rate"))
    .orderBy(F.min("vehicle_flight_num")))

In [0]:
import matplotlib.pyplot as plt

curve = (gold
    .withColumn("flight_bucket",
        F.when(F.col("vehicle_flight_num") == 1, "1")
         .when(F.col("vehicle_flight_num") <= 3, "2–3")
         .when(F.col("vehicle_flight_num") <= 10, "4–10")
         .when(F.col("vehicle_flight_num") <= 30, "11–30")
         .otherwise("31+"))
    .groupBy("flight_bucket")
    .agg(F.count("*").alias("launches"),
         F.avg(F.col("failed").cast("int")).alias("failure_rate"),
         F.min("vehicle_flight_num").alias("_sort"))
    .orderBy("_sort")
    .toPandas())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(curve["flight_bucket"], curve["failure_rate"] * 100, color="#4C72B0")

for bar, rate, n in zip(bars, curve["failure_rate"], curve["launches"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{rate:.1%}\n(n={n:,})", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Vehicle flight number")
ax.set_ylabel("Failure rate (%)")
ax.set_title("Launch failure rate falls as rockets gain experience")
ax.set_ylim(0, curve["failure_rate"].max() * 100 * 1.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

RQ1: Does failure risk drop as a rocket gains experience?

What I did

- Gave every launch a vehicle flight number: its position in that rocket type’s history (1 = the rocket’s first ever launch, 2 = its second, and so on)
- Counted only earlier launches, so each launch’s number reflects experience before it flew
- Grouped launches into buckets by flight number, since individual flight numbers have too few launches to compare
- Calculated the share of launches that failed in each bucket

Results

- Maiden flights: 23.9% failed (n=440)
- Flights 2–3: 14.2% (n=641)
- Flights 4–10: 9.5% (n=1,248)
- Flights 11–30: 6.6% (n=1,444)
- Flights 31+: 2.6% (n=3,634)

## RQ2: Do commercial providers mature faster than government programs?

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# commercial vs. government (civil + military); academic is set aside (few launches)
rq2 = (flight_bucket(gold)
    .withColumn("provider_group",
        F.when(F.col("provider_class") == "commercial", "Commercial")
         .when(F.col("is_government"), "Government"))
    .filter(F.col("provider_group").isNotNull())
    .groupBy("provider_group", "flight_bucket")
    .agg(F.count("*").alias("n"),
         F.sum(F.col("failed").cast("int")).alias("fails"),
         F.min("vehicle_flight_num").alias("_sort"))
    .orderBy("provider_group", "_sort")
    .toPandas())

# 95% Wilson confidence intervals
z, n, p = 1.96, rq2["n"], rq2["fails"] / rq2["n"]
center = (p + z**2/(2*n)) / (1 + z**2/n)
half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / (1 + z**2/n)
rq2["rate"], rq2["lo"], rq2["hi"] = p, center - half, center + half

display(rq2[["provider_group", "flight_bucket", "n", "fails", "rate"]])

# chart
buckets = ["1", "2-3", "4-10", "11-30", "31+"]
x = np.arange(len(buckets))
width = 0.38

fig, ax = plt.subplots(figsize=(7, 4))
for i, (group, color) in enumerate([("Government", "#4C72B0"), ("Commercial", "#DD8452")]):
    g = rq2[rq2["provider_group"] == group].set_index("flight_bucket").reindex(buckets)
    pos = x + (i - 0.5) * width
    ax.bar(pos, g["rate"] * 100, width, color=color, label=group)
    ax.errorbar(pos, g["rate"] * 100,
                yerr=[(g["rate"] - g["lo"]) * 100, (g["hi"] - g["rate"]) * 100],
                fmt="none", ecolor="black", capsize=3, linewidth=1)

ax.set_xticks(x, buckets)
ax.set_xlabel("Vehicle flight number")
ax.set_ylabel("Failure rate (%)")
ax.set_title("Learning curves: commercial vs. government providers")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
decade = (gold
    .withColumn("decade", (F.floor(F.col("launch_year") / 10) * 10).cast("int"))
    .withColumn("provider_group",
        F.when(F.col("provider_class") == "commercial", "Commercial")
         .when(F.col("is_government"), "Government"))
    .groupBy("decade", "provider_group")
    .agg(F.count("*").alias("n"),
         F.sum(F.col("failed").cast("int")).alias("fails"))
    .withColumn("rate", F.round(F.col("fails") / F.col("n"), 3))
    .orderBy("decade", "provider_group")
    .toPandas())

display(decade)

In [0]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for group, color in [("Government", "#4C72B0"), ("Commercial", "#DD8452")]:
    g = decade[decade["provider_group"] == group]
    ax1.plot(g["decade"], g["rate"] * 100, marker="o", color=color, label=group)
    ax2.bar(g["decade"] + (-2 if group == "Government" else 2), g["n"], width=4,
            color=color, label=group)

ax1.set_title("Failure rate by decade")
ax1.set_xlabel("Decade")
ax1.set_ylabel("Failure rate (%)")
ax1.legend(frameon=False)

ax2.set_title("Number of launches by decade")
ax2.set_xlabel("Decade")
ax2.set_ylabel("Launches")
ax2.legend(frameon=False)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

The commercial advantage is recent, not constant.

In [0]:
display(gold
    .filter(F.col("provider_class") == "commercial")
    .groupBy("lv_family")
    .agg(F.count("*").alias("launches"),
         F.sum(F.col("failed").cast("int")).alias("fails"))
    .withColumn("share_of_commercial",
        F.round(F.col("launches") / F.sum("launches").over(Window.partitionBy()), 3))
    .orderBy(F.desc("launches"))
    .limit(10))

In [0]:
display(flight_bucket(gold.filter(F.col("lv_family") != "Falcon9"))
    .withColumn("provider_group",
        F.when(F.col("provider_class") == "commercial", "Commercial")
         .when(F.col("is_government"), "Government"))
    .filter(F.col("provider_group").isNotNull())
    .groupBy("provider_group", "flight_bucket")
    .agg(F.count("*").alias("n"),
         F.round(F.avg(F.col("failed").cast("int")), 3).alias("rate"),
         F.min("vehicle_flight_num").alias("_sort"))
    .orderBy("provider_group", "_sort")
    .drop("_sort"))

still mature faster

What I did

- Split launches into commercial (run by a business) and government (run by a civil agency or military)
- Compared failure rates by flight number for each group
- Checked whether the results hold up when accounting for decade and removing Falcon 9

Results

- First flights are about equally risky: 21% commercial vs. 25% government
- Commercial rockets improve faster: by flights 11–30, 2.7% vs. 8.8%
- The gap only appears from the 2010s on; before that, both groups were about equally reliable
- Falcon 9 is 31% of commercial launches; removing it doesn’t change the early results, but mature commercial rockets look less exceptional (0.8% → 2.0%)

## RQ3: Which design and mission factors carry the most risk?

In [0]:
def rate_by(df, col):
    return (df.groupBy(col)
        .agg(F.count("*").alias("launches"),
             F.sum(F.col("failed").cast("int")).alias("fails"),
             F.round(F.avg(F.col("failed").cast("int")) * 100, 1).alias("failure_pct"))
        .orderBy(col))

In [0]:
# stages and strap-on boosters
display(rate_by(gold, "n_core_stages"))
display(rate_by(gold, "has_strapons"))

Stages: Rockets are built from stacked stages that fire in sequence and drop away when empty, so each added stage means another separation and engine ignition that must succeed. Failure rates rise with stage count beyond two (5.2% for two stages, 8.1% for three, 12.3% for four), suggesting added complexity increases risk.

Strap-on boosters: Strap-on boosters are extra rockets attached to the sides of the first stage that add thrust at liftoff and then detach. Rockets with boosters failed less often (5.7% vs. 8.1%), likely because boosters are common on large, mature heavy-lift rockets with long flight records rather than because boosters themselves reduce risk.

In [0]:
# rocket size
from pyspark.ml.feature import QuantileDiscretizer

sized = QuantileDiscretizer(numBuckets=4, inputCol="launch_mass_t",
                            outputCol="mass_quartile").fit(gold).transform(gold)

display(sized.groupBy("mass_quartile")
    .agg(F.min("launch_mass_t").alias("min_mass_t"),
         F.max("launch_mass_t").alias("max_mass_t"),
         F.count("*").alias("launches"),
         F.round(F.avg(F.col("failed").cast("int")) * 100, 1).alias("failure_pct"))
    .orderBy("mass_quartile"))

Rocket size: Launch mass measures how heavy a rocket is at liftoff, which reflects its overall size and power. The smallest quarter of rockets (under ~140 tonnes) failed 13.6% of the time, about three times the 4–5% of larger rockets, while size made little difference above that threshold, likely because small rockets are more often new, low-budget, or early-era designs.

In [0]:
# mission factors
display(rate_by(gold, "launch_profile").orderBy(F.desc("launches")).limit(15))
display(rate_by(gold, "agency_country").orderBy(F.desc("launches")).limit(10))

https://www.planet4589.org/space/gcat/web/intro/profile.html
Target orbit: where the payload was headed. LEO (low Earth orbit), GTO (transfer to geostationary), SSO (sun-synchronous, used by imaging satellites), ISS, LSS (other space stations), MOL (Molniya, a long elliptical orbit Russia uses), GEO (direct geostationary)
Stage disposal: what was planned for the spent upper stage. O = left in orbit, D = deliberately deorbited, P/L = orbit lowered so it decays faster, A = stays attached to the payload, S = dropped just short of orbit for a targeted reentry, R = recovered (Space Shuttle)


What the results suggest
- Station missions are the safest: ISS 2.5%, other stations 1.5%. These are mostly crewed and cargo flights, which use the most proven, human-rated rockets.
- Standard orbits sit around the average: LEO 8.1%, GTO 7.0%, SSO 10.0% (with stage left in orbit).
- Stage disposal codes mostly track era, not risk. Deorbiting or lowering spent stages (D, P, L) is a modern debris-mitigation practice, so those launches are mostly recent and on mature rockets. That’s why “Sat LEO D” shows just 0.9%. By the same logic, “A” (stage attached to payload, 14.7%) is mostly an older practice.

In [0]:
# failure rate by target orbit
gold_orbit = gold.withColumn("target_orbit", F.split("launch_profile", " ").getItem(1))

display(rate_by(gold_orbit, "target_orbit").orderBy(F.desc("launches")))

| Target orbit | Description | Launches | Failures | Failure rate |
|---|---|---|---|---|
| EEO | Earth escape: Moon, planets, deep space | 235 | 51 | 21.7% |
| MEO | Medium Earth orbit: navigation satellites (GPS, GLONASS) | 193 | 23 | 11.9% |
| DSO | Deep space orbit: very high Earth orbits | 61 | 5 | 8.2% |
| GEO | Direct insertion into geostationary orbit | 288 | 21 | 7.3% |
| LEO | Low Earth orbit | 3,717 | 261 | 7.0% |
| GTO | Geosynchronous transfer orbit | 903 | 61 | 6.8% |
| SSO | Sun-synchronous orbit: imaging satellites | 963 | 60 | 6.2% |
| MOL | Molniya: long elliptical orbits used by Russia | 306 | 13 | 4.2% |
| MTO | Medium Earth transfer orbit | 80 | 2 | 2.5% |
| ISS | International Space Station | 312 | 6 | 1.9% |
| LSS | Other low Earth orbit space stations | 224 | 3 | 1.3% |
| STO | Supersynchronous transfer orbit | 87 | 1 | 1.1% |
| HEO | Highly elliptical orbit | 37 | 0 | 0.0% |
| Test | Test flight | 1 | 0 | 0.0% |

In [0]:
display(rate_by(gold, "agency_country").orderBy(F.desc("launches")).limit(10))

- Launching country = home country of the provider running the launch
- Major programs (US, USSR/Russia, China, Europe) fail 4–7% of the time
- Smaller programs fail more: Japan 13%, India 12%, Iran 48% (21 launches)
- Country mostly reflects experience and rocket family, so it may matter less once those are in the model